# What to do when you get an error

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [2]:
!pip install datasets evaluate transformers[sentencepiece]
!apt install git-lfs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.6 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git-lfs is already the newest version (3.0.2-1ubuntu0.3).
0 upgraded, 0 newly installed, 0 to remove and 100 not upgraded.


You will need to setup git, adapt your email and name in the following cell.

In [8]:
!git config --global user.email "pvbcod@gmail.com"
!git config --global user.name "pvbgeek"

You will also need to be logged in to the Hugging Face Hub. Execute the following and enter your credentials.

In [29]:
from huggingface_hub import notebook_login

notebook_login()

In [30]:
from huggingface_hub import notebook_login

notebook_login()

In [17]:
# from distutils.dir_util import copy_tree
# from huggingface_hub import Repository, snapshot_download, create_repo, get_full_repo_name


# def copy_repository_template():
#     # Clone the repo and extract the local path
#     template_repo_id = "lewtun/distilbert-base-uncased-finetuned-squad-d5716d28"
#     commit_hash = "be3eaffc28669d7932492681cd5f3e8905e358b4"
#     template_repo_dir = snapshot_download(template_repo_id, revision=commit_hash)
#     # Create an empty repo on the Hub
#     model_name = template_repo_id.split("/")[1]
#     create_repo(model_name, exist_ok=True)
#     # Clone the empty repo
#     new_repo_id = get_full_repo_name(model_name)
#     new_repo_dir = model_name
#     repo = Repository(local_dir=new_repo_dir, clone_from=new_repo_id)
#     # Copy files
#     copy_tree(template_repo_dir, new_repo_dir)
#     # Push to Hub
#     repo.push_to_hub()

from huggingface_hub import snapshot_download, create_repo, get_full_repo_name, upload_folder

def copy_repository_template():
    template_repo_id = "lewtun/distilbert-base-uncased-finetuned-squad-d5716d28"
    commit_hash = "be3eaffc28669d7932492681cd5f3e8905e358b4"
    template_repo_dir = snapshot_download(template_repo_id, revision=commit_hash)
    model_name = template_repo_id.split("/")[1]
    create_repo(model_name, exist_ok=True)
    new_repo_id = get_full_repo_name(model_name)
    # Upload directly from the snapshot folder — no local clone needed
    upload_folder(
        folder_path=template_repo_dir,
        repo_id=new_repo_id,
        commit_message="Upload model from template"
    )

In [31]:
from huggingface_hub import create_repo, get_full_repo_name
from transformers import pipeline, AutoTokenizer, AutoModelForQuestionAnswering

# Step 1 — Define the function (FIXED)
def copy_repository_template():
    template_repo_id = "lewtun/distilbert-base-uncased-finetuned-squad-d5716d28"
    model_name = template_repo_id.split("/")[1]
    new_repo_id = get_full_repo_name(model_name)

    # 1. Create the repository if it doesn't exist
    create_repo(new_repo_id, exist_ok=True)

    # 2. Load the model and tokenizer from the original template
    #    This will download the necessary files (pytorch_model.bin, config.json, tokenizer files)
    print(f"Loading model and tokenizer from {template_repo_id}...")
    tokenizer = AutoTokenizer.from_pretrained(template_repo_id)
    model = AutoModelForQuestionAnswering.from_pretrained(template_repo_id)
    print("Model and tokenizer loaded.")

    # 3. Push the loaded model and tokenizer to the new repository
    print(f"Pushing model and tokenizer to {new_repo_id}...")
    tokenizer.push_to_hub(new_repo_id, commit_message="Initial upload of tokenizer")
    model.push_to_hub(new_repo_id, commit_message="Initial upload of model")
    print("Model and tokenizer pushed to Hub.")

# Step 2 — CALL it
copy_repository_template()

# Step 3 — Only now will this work
model_checkpoint = get_full_repo_name("distilbert-base-uncased-finetuned-squad-d5716d28")
reader = pipeline("question-answering", model=model_checkpoint)

Loading model and tokenizer from lewtun/distilbert-base-uncased-finetuned-squad-d5716d28...


tokenizer_config.json:   0%|          | 0.00/258 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/265M [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/563 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/265M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/322 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [32]:
from huggingface_hub import list_repo_files

list_repo_files(repo_id=model_checkpoint)

['.gitattributes',
 'README.md',
 'config.json',
 'model.safetensors',
 'tokenizer.json',
 'tokenizer_config.json']

In [33]:
from transformers import AutoConfig

pretrained_checkpoint = "distilbert-base-uncased"
config = AutoConfig.from_pretrained(pretrained_checkpoint)

In [34]:
config.push_to_hub(model_checkpoint, commit_message="Add config.json")

CommitInfo(commit_url='https://huggingface.co/ParthGeek/distilbert-base-uncased-finetuned-squad-d5716d28/commit/7a7d8a4fa667e94830f06c7a8ffc6320b3e0c6ce', commit_message='Add config.json', commit_description='', oid='7a7d8a4fa667e94830f06c7a8ffc6320b3e0c6ce', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ParthGeek/distilbert-base-uncased-finetuned-squad-d5716d28', endpoint='https://huggingface.co', repo_type='model', repo_id='ParthGeek/distilbert-base-uncased-finetuned-squad-d5716d28'), pr_revision=None, pr_num=None)

In [35]:
reader = pipeline("question-answering", model=model_checkpoint, revision="main")

context = r"""
Extractive Question Answering is the task of extracting an answer from a text
given a question. An example of a question answering dataset is the SQuAD
dataset, which is entirely based on that task. If you would like to fine-tune a
model on a SQuAD task, you may leverage the
examples/pytorch/question-answering/run_squad.py script.

🤗 Transformers is interoperable with the PyTorch, TensorFlow, and JAX
frameworks, so you can use your favourite tools for a wide variety of tasks!
"""

question = "What is extractive question answering?"
reader(question=question, context=context)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

{'score': 0.3761069178581238,
 'start': 34,
 'end': 95,
 'answer': 'the task of extracting an answer from a text\ngiven a question'}

In [36]:
tokenizer = reader.tokenizer
model = reader.model

In [37]:
question = "Which frameworks can I use?"

In [39]:
import torch

inputs = tokenizer(question, context, add_special_tokens=True, return_tensors="pt")
# Move input tensors to the same device as the model
inputs = {k: v.to(model.device) for k, v in inputs.items()}

input_ids = inputs["input_ids"][0]
outputs = model(**inputs)
answer_start_scores = outputs.start_logits
answer_end_scores = outputs.end_logits
# Get the most likely beginning of answer with the argmax of the score
answer_start = torch.argmax(answer_start_scores)
# Get the most likely end of answer with the argmax of the score
answer_end = torch.argmax(answer_end_scores) + 1
answer = tokenizer.convert_tokens_to_string(
    tokenizer.convert_ids_to_tokens(input_ids[answer_start:answer_end])
)
print(f"Question: {question}")
print(f"Answer: {answer}")

Question: Which frameworks can I use?
Answer: pytorch, tensorflow, and jax


In [40]:
# inputs["input_ids"][:5]

inputs["input_ids"][0][:5]   # [0] to get first sequence, then [:5] for first 5 tokens

tensor([ 101, 2029, 7705, 2015, 2064], device='cuda:0')

In [41]:
type(inputs["input_ids"])

torch.Tensor